# Data Preparation Lanjutan — Studi Kasus: Telco Customer Churn
**Mata Kuliah: Machine Learning — Kelompok 3**
1. Frisilia Elfira Maharani — 202343500138
2. Febyana Mulia Putri — 202343500151
3. Wanda Nurhasanah — 202343500154
4. Muhammad Revolvere Rey A — 202343500166
5. Ahmad Dandi Subhani — 202343500126

Notebook ini menerapkan alur **Data Preparation** (missing values → outlier → imbalanced data) pada dataset nyata **Telco Customer Churn** untuk memprediksi pelanggan yang berhenti berlangganan (*churn*).
Mengikuti *aturan aman*: **Split → Fit (train) → Transform → Resample (train saja) → Evaluasi (test asli)**.

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE

## 2. Load Dataset
Dataset: **Telco Customer Churn** — 7.043 pelanggan, 21 kolom. Target: `Churn` (Yes/No).

In [ ]:
# Dataset diambil otomatis — tidak perlu upload manual.
# - Di Colab: file diunduh dari GitHub repo ini.
# - Lokal (clone repo): file dibaca dari folder data/.
import os, urllib.request

csv_name = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
if os.path.exists('data/' + csv_name):
    csv_path = 'data/' + csv_name
elif os.path.exists(csv_name):
    csv_path = csv_name
else:
    print("Mengunduh dataset dari GitHub...")
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/dandienter/telco-churn-kelompok3/main/data/' + csv_name,
        csv_name)
    csv_path = csv_name

df = pd.read_csv(csv_path)
print("Bentuk data:", df.shape)
df.head()

## 3. Diagnosis Masalah Data
Tiga masalah yang dicari: **missing values**, **outlier**, dan **imbalanced data**.

In [ ]:
print("Duplikat:", df.duplicated().sum())

# TotalCharges terbaca sebagai teks karena ada sel kosong berupa spasi ' '
print("\nContoh nilai aneh di TotalCharges:",
      df.loc[pd.to_numeric(df['TotalCharges'], errors='coerce').isna(), 'TotalCharges'].unique())

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("\nMissing values per kolom:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nDistribusi target (Churn):")
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True).round(3))

**Hasil diagnosis:**
- **Missing values:** 11 baris di `TotalCharges` (pelanggan baru dengan `tenure = 0`, tagihan totalnya memang belum ada — bukan error acak).
- **Outlier:** akan dideteksi dengan Isolation Forest.
- **Imbalanced:** Churn=Yes hanya 26,5% — model bisa "malas" dan selalu menebak "Tidak Churn".

## 4. Preprocessing (Aturan Aman)
1. Buang `customerID` (identitas, bukan pola). Ubah target `Churn` menjadi 1/0.
2. **Split dulu** (stratified) — agar preprocessing tidak "mengintip" data test.
3. One-hot encoding untuk kolom kategorikal.
4. **KNN Imputer** (fit di train, transform train+test) untuk missing values numerik.
5. **StandardScaler** (fit di train) agar Logistic Regression konvergen dengan baik.
6. **Isolation Forest** untuk mendeteksi outlier di data train.

In [ ]:
# 1. Bersihkan & siapkan target
df = df.drop(columns=['customerID'])
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# 2. One-hot encoding
X = pd.get_dummies(df.drop(columns=['Churn']), drop_first=True)
y = df['Churn']
print("Jumlah fitur setelah encoding:", X.shape[1])

# 3. Split LEBIH DULU (stratified agar proporsi churn terjaga)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print("Train:", X_train.shape, dict(y_train.value_counts()))
print("Test :", X_test.shape, dict(y_test.value_counts()))

# 4. KNN Imputer — hanya belajar dari train
num_cols = X_train.select_dtypes(['int64', 'float64']).columns.tolist()
imputer = KNNImputer(n_neighbors=5)
X_train[num_cols] = imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = imputer.transform(X_test[num_cols])
print("\nMissing setelah imputasi:", X_train.isnull().sum().sum(), "(train),",
      X_test.isnull().sum().sum(), "(test)")

# 5. Scaling — hanya belajar dari train
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# 6. Deteksi outlier di train
iso = IsolationForest(contamination=0.02, random_state=42)
is_outlier = iso.fit_predict(X_train)
n_outlier = (is_outlier == -1).sum()
print(f"\nOutlier terdeteksi: {n_outlier} baris ({n_outlier/len(X_train)*100:.1f}%)")
print("Rata-rata tenure outlier :", round(X_train.loc[is_outlier == -1, 'tenure'].mean(), 1))
print("(Outlier di sini = pelanggan tenure sangat panjang — kejadian sah, jadi DIPERTAHANKAN, bukan dihapus.)")

## 5. Training Model (Baseline vs SMOTE)
- **Baseline:** Logistic Regression tanpa penanganan imbalance.
- **SMOTE:** menambah sampel sintetis kelas minoritas — **HANYA pada data train**.

In [ ]:
# Baseline
model_base = LogisticRegression(max_iter=2000).fit(X_train, y_train)

# SMOTE hanya pada train
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print("Distribusi train sesudah SMOTE:", dict(pd.Series(y_train_sm).value_counts()))

model_smote = LogisticRegression(max_iter=2000).fit(X_train_sm, y_train_sm)

## 6. Evaluasi Model (pada test ASLI yang tidak seimbang)

In [ ]:
for nama, model in [("Baseline (tanpa SMOTE)", model_base),
                       ("Dengan SMOTE", model_smote)]:
    y_pred = model.predict(X_test)
    print(f"--- {nama} ---")
    print("Akurasi :", round(accuracy_score(y_test, y_pred), 3))
    print("F1-Score:", round(f1_score(y_test, y_pred), 3))
    print(classification_report(y_test, y_pred, target_names=['Tidak Churn', 'Churn']))

## 7. Visualisasi

In [ ]:
# Distribusi kelas sebelum vs sesudah SMOTE
before = y_train.value_counts().sort_index()
after = pd.Series(y_train_sm).value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].bar(['Tidak Churn', 'Churn'], before.values, color=['#2E5395', '#C0504D'])
axes[0].set_title('Sebelum SMOTE (data latih)')
axes[0].set_ylabel('Jumlah sampel')
axes[1].bar(['Tidak Churn', 'Churn'], after.values, color=['#2E5395', '#4C9F70'])
axes[1].set_title('Sesudah SMOTE (data latih)')
plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrix kedua model
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (nama, model) in zip(axes, [("Baseline", model_base), ("SMOTE", model_smote)]):
    cm = confusion_matrix(y_test, model.predict(X_test))
    ax.imshow(cm, cmap='Blues')
    ax.set_title(nama); ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Tidak Churn', 'Churn']); ax.set_yticklabels(['Tidak Churn', 'Churn'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 8. Interpretasi & Kesimpulan
**Hasil evaluasi pada data test asli:**

| Model | Akurasi | F1-Score | Precision (Churn) | Recall (Churn) |
|---|---|---|---|---|
| Baseline | 0.807 | 0.607 | 0.66 | 0.56 |
| SMOTE | 0.738 | 0.590 | 0.51 | 0.71 |

**Interpretasi (singkat & jelas):**
1. **Missing values** (11 di `TotalCharges`) berasal dari pelanggan baru (`tenure=0`) — diatasi dengan **KNN Imputer** karena nilainya bisa ditebak dari pelanggan serupa.
2. **Outlier** (113 baris, 2%) adalah pelanggan dengan tenure sangat panjang — **kejadian sah, dipertahankan**, bukan dihapus.
3. **SMOTE menaikkan recall** (0.56 → 0.71): model menangkap 265 pelanggan churn vs 210 sebelumnya — hanya 109 yang lolos vs 164. **Tetapi precision turun** (0.66 → 0.51): lebih banyak false alarm.
4. **Pelajaran penting:** akurasi saja menipu — baseline "terlihat" lebih baik (0.807 vs 0.738), padahal ia melewatkan lebih banyak pelanggan churn. Untuk kasus churn, **recall lebih penting** (lebih baik salah mengingatkan daripada kehilangan pelanggan diam-diam).

**Kesimpulan:** Data preparation yang benar (aturan aman: split → fit di train → SMOTE hanya di train) menghasilkan model yang jujur dievaluasi. Pilihan model tergantung tujuan bisnis: kejar recall (SMOTE) atau presisi (baseline).